# Lab 9 — Let an AI attack your project
### EP0418 · Topic 9 · 2 practical hours

**You need:** your Lab 8 notebook (in your Google Drive) and the printed Lab 9
sheet. This notebook fetches its own files.

**Time plan (2 hours):** Part 1, two checks: 20 min · Task 1, predict and run the critic: 20 min ·
Task 2, the findings table: 30 min · Task 3, fixes: 30 min · wrap-up: 20 min.

---
Today you point an AI assistant at your own pipeline and ask it to **find what is
wrong with it**.

> **What is assessed is your rebuttal, not what the critic finds.**

The critic will raise things that are wrong. Saying *why* they are wrong, with
evidence, is worth exactly as much as fixing the ones that are right.

**If Gemini stops answering.** Free Colab Gemini has a daily limit. The circle
beside the model name in the chat box shows how much you have used; at 100% it says
*Daily limit reached*. Then open ChatGPT, Claude or Copilot in a new browser tab and
paste the same prompt there; copy any code it writes into a new cell (**+ Code** at
the top of the page) and run it. It cannot see your notebook, so if a check fails or
the code shows red error text, paste that output into its chat too.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. If a box *Review custom instructions* appears while you save the copy, click **Make a copy**: it keeps the lab's short rules for the AI assistant. Colab opens your copy in a new tab with *Copy of* at the start of its name at the top left: work in that tab and close the old one. Reopen your copy from Google Drive next time.

> **Then run the grey code cell just below** (click in it and press Shift+Enter, or click the play button at its left). Colab may warn that the notebook was not authored by Google: click **Run anyway**. You should see nothing in red; lines like *fetched ...*, a row count or a short table are normal. That means the course files and tools are loaded. Every lab starts this way. From here on, **run every grey code cell when you reach it**, top to bottom. **To write in a text box** (a cell with a bold label ending in a colon), double-click it, type, then press Shift+Enter.

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check, show


def keep(name):
    """The value Gemini's cell already made under this name, or ... if there is none yet."""
    global _NAMED
    v = globals().get(name, ...)
    if v is ... or v is None:
        print(f"Not there yet: nothing called {name}. Not run Gemini's green cell yet? Click Accept and "
              f"run on it, then run this cell again. Already ran it? Type in the same chat: call the "
              f"result {name}. Then Accept and run, and run this cell again.")
        _NAMED = True
        return ...
    return v


def filled(*values):
    """True once every TODO above has a value. Otherwise says what to do."""
    global _NAMED
    if any(v is ... or v is None for v in values):
        if not globals().get("_NAMED"):      # keep() has already said exactly what is missing
            print("Not there yet. Run Gemini's code first (click Accept and run on its green cell, "
                  "or paste its code into a new cell and run it), then run this cell again.")
        _NAMED = False
        return False
    _NAMED = False
    return True


def show_predict(data, unit):
    """Prediction League: run the newest predict that Gemini wrote and print its code to copy."""
    ran = globals().get("In", [])                                   # every cell you have run, oldest first
    mine = [c for c in ran if "def predict(" in c and "The baseline predicts" not in c and "show_predict" not in c]
    if not mine:
        print("Gemini's predict has not run yet. Find the cell Gemini added in Step 2 (it starts with")
        print("def predict, or with import lines just above it), click in it, press Shift+Enter,")
        print("then run this cell again."); return
    code = mine[-1].strip()
    for short, line in (("np.", "import numpy as np"), ("pd.", "import pandas as pd")):
        if short in code and line not in code:                     # the league page starts empty
            code = line + "\n" + code
    ns = {}
    exec(code, ns)
    print("Your predict says", round(ns["predict"](data.copy()), 1), unit)
    print()
    print("----- copy from the line below -----")
    print(code)
    print("----- to the line above -----")

# The Part 1 cells need these files beside the notebook.
fetch("smart_building_sensors.csv", "smart_self_daily.csv", "smart_money_ohlcv.csv")

## Part 1A — Check the zeros: a real zero, or nothing recorded · `VERIFY`

Part 1 runs two checks that Labs 1 to 8 did not use. **Your project's data might
need them**: zeros that mean nothing was recorded (Part 1A), and lookahead bias,
which means using information you would not have had at the time (Part 1B). Tags show where you are in the loop: `PROMPT` → `GENERATE` → `VERIFY` → `REFINE`.

**What each field means.** Part 1 uses three of the course's project files. Only
these columns matter today.

| File | Field | What it means | Example |
|---|---|---|---|
| `smart_building_sensors.csv` | `timestamp` | when the reading was taken: every 5 minutes, 1 May to 29 June 2026 | 2026-05-01 00:05 |
| `smart_building_sensors.csv` | `occupancy` | how many people were in the classroom | 0, 28 |
| `smart_building_sensors.csv` | `co2_ppm`, `temp_c` | carbon dioxide in parts per million; temperature in °C | 420, 24.9 |
| `smart_self_daily.csv` | `date` | one day of one person's smartwatch data, 177 days in all | 2026-01-20 |
| `smart_self_daily.csv` | `steps` | steps the watch counted that day | 7689 |
| `smart_self_daily.csv` | `active_minutes`, `resting_hr` | minutes of exercise; resting heart rate in beats per minute, blank if not measured | 46, 66.9 |
| `smart_money_ohlcv.csv` | `Date`, `Close` | one trading day, and the price when the market closed that day | 2023-01-03, 2.84 |

1. **Run the cell below.**
2. `VERIFY` **Read the report.** You should see:
   - **occupancy:** PASS, 12,275 zero values, within the 20,000 the cell allowed;
   - **steps:** NOTE, 11 rows with steps exactly 0;
   - **no_missing:** PASS, no blank values in `co2_ppm` and `temp_c`.

In [ ]:
# --- zeros_are_real: is a zero a measurement, or a device that was off? -------
# In the building data, occupancy = 0 means the room was empty. A REAL zero.
b = pd.read_csv("smart_building_sensors.csv", parse_dates=["timestamp"])
check.reset()
check.zeros_are_real(b, "occupancy", max_zeros=20000)      # passes — real zeros

# In personal activity data, steps = 0 usually means the watch was in a drawer.
# Same value, completely different meaning.
if os.path.exists("smart_self_daily.csv"):
    s = pd.read_csv("smart_self_daily.csv", parse_dates=["date"])
    check.zeros_are_real(s, "steps")                       # NOTE — investigate

# --- no_missing: are there blanks left in what you are about to analyse? ------
check.no_missing(b, cols=["co2_ppm", "temp_c"])

check.report()

**What the cell above just checked**, on the real files (it plays slowly, about a minute):
![Occupancy zeros fall at night, when the room is empty: real zeros. The 11 zero-step days also have 0 active minutes and no heart rate: the watch was off, so those zeros are missing](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab09_zeros.gif)

### Read those two `zeros_are_real` results

`occupancy = 0` **passed**. `steps = 0` came back as a **NOTE**.

The check cannot tell which is which by itself: the cell told it, with
`max_zeros=20000` for occupancy, that many zeros are expected there. The 12,275
occupancy zeros all fall at night or at weekends: the room really was empty. The 11
zero-step days are different: on each one `active_minutes` is also 0 and
`resting_hr` is blank, so the watch was not worn. A person with a watch in a drawer
did not really take zero steps. Averaged in as 0, those 11 days pull the mean of
`steps` down from about 8,370 to about 7,850.

**Write in the box below** any column in your project's data where zero might mean
"not recorded", and whether you will treat its zeros as real or as missing. If you
find no such column, write *none found*; if you do not know your columns yet, write
*check when the data is loaded*. That line goes into the data section of your CA3
report.

**Column where zero might mean "not recorded", and what I will do with its zeros:**

*(Your answer box: double-click this cell. Type your answer here.)*

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

This depends on your own project data, so no single column is right. A good answer
names the column, says what a zero in it means, and says what you will do: keep the
zeros as real, or turn them into NaN before any average. Example, from the file the
Part 1A check cell read: `steps` has 11 zeros in 177 days. On every one of those days
`active_minutes` is also 0 and `resting_hr` is blank, so the watch was off. Those
zeros are missing, not real. Left in, they pull the mean from about 8,370 down to
about 7,850. The common weak answer is *none found* written without looking at the
data. Counts, sales, rainfall and steps can all hide zeros that mean nothing was
recorded.
</details>

## Part 1B — Check a signal for lookahead bias · `VERIFY`

**Lookahead bias** means using information you would not have had at the time. If
you compute a signal from today's closing price and act on it today, you used a
price that is only known when the market shuts. This is the most common way a
student backtest produces an impressive and completely fake result. It matters most
for Track 1 (Smart Money) and for any project that acts on a signal, but everyone
runs it.

The cell below builds a signal from `smart_money_ohlcv.csv`: 1 on days the close is
above its average over the last 20 trading days, otherwise 0.

1. `VERIFY` **Run the cell below.** You should see two reports:
   - **Report 1:** `signal_is_shifted` FAIL, 107 rows differ from the signal shifted by 1;
   - **Report 2:** `signal_is_shifted` PASS.

In [ ]:
if os.path.exists("smart_money_ohlcv.csv"):
    m = pd.read_csv("smart_money_ohlcv.csv", parse_dates=["Date"])
    raw_signal = (m.Close > m.Close.rolling(20).mean()).astype(int)

    check.reset()
    check.signal_is_shifted(raw_signal, raw_signal, by=1)        # FAILS - lookahead
    check.report()
    check.signal_is_shifted(raw_signal, raw_signal.shift(1), by=1)  # PASSES
    check.report()

**What the cell above just checked**, on four real trading days (it plays slowly, about a minute):
![The signal for 10 March 2023 uses that day's close, which is only known when the market shuts. shift(1) uses it on the next trading day instead. A backtest with the unshifted signal turns 1 dollar into 3.60, a fake result; shifted, 1.47](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab09_lookahead.gif)

The first call fails and the second passes, on the same data. The only difference
is one `.shift(1)`: acting on yesterday's signal instead of today's. The 107 rows in
the FAIL are the days the signal changed; only on those days do the two versions
differ.

Nothing to do here now. If your project triggers an action from a signal, run
`check.signal_is_shifted` on your own signal in your project notebook, the way this
cell does, and keep the PASS as evidence for CA3.

## Task 1 — Predict, then run the critic · `PROMPT` `VERIFY`

*Topic 9: the assistant runs checks; you judge them. A second assistant, in a fresh
chat, is asked to find faults. It finds real faults, and some that are not.*

Task 1 has two parts: **1A**, predict what the critic will find; **1B**, run it in
your Lab 8 notebook.

### 1A · Predict before the critic runs

The critic checks your Lab 8 pipeline for nine faults, called **probes**:

| # | Probe | In plain words |
|---|---|---|
| 1 | Silent row loss | rows disappear and nothing says how many |
| 2 | Sentinel values | a code such as -999, meaning "no reading", is used as a number |
| 3 | Zero vs missing | a 0 that means "nothing recorded" is treated as a real 0 (Part 1A) |
| 4 | Time index | rows are not evenly spaced in time before `.shift()` moves them |
| 5 | Horizon mismatch | the model and the baseline are scored on predicting different times ahead |
| 6 | Lookahead bias | a signal uses information from the future (Part 1B) |
| 7 | Baseline honesty | the baseline is weaker than the simple rule a sensible person would use |
| 8 | Response layer | the alert repeats too often: no debounce, no limit |
| 9 | Failure mode | what the pipeline does when the data source returns nothing |

Write two short answers in the box below:

1. Which probe do you think your pipeline is most likely to FAIL, and why? Name the
   stage: ingest, clean, analyse, decide or respond.
2. Which probe do you think the critic may get wrong about your pipeline, and what in
   your Lab 8 notebook would show it?

**My plan:**

*(Your answer box: double-click this cell. Type your answer here.)*

### 1B · Run the critic in your Lab 8 notebook

The critic needs to see your pipeline. Gemini in Colab reads the notebook it is
open in, so you run the critic **inside your Lab 8 notebook**.

1. **Copy the prompt** in the grey box below first: hover over the box and click the
   copy icon at its top right. If there is no copy icon, click just before *Rules:*,
   drag down to the end of *defined in this notebook.*, and press Ctrl+C (Cmd+C on a
   Mac). Do not press Ctrl+A: that selects the whole page. Its last line names
   the functions ingest, clean, analyse, decide, respond and pipeline: those are
   the names Lab 8 used. If you renamed any, edit that line in the chat box after
   pasting and before pressing Enter.
2. **Open your Lab 8 notebook** in a new browser tab: in Google Drive, open the
   folder *Colab Notebooks* and double-click *Copy of Lab08_End_to_End_STUB* (or
   whatever you renamed it to).
3. **Run it all:** menu **Runtime → Run all**, so your functions exist. It is done
   when every grey code cell shows a number in square brackets on its left (text
   cells never do). If a cell waits for
   a file upload, click **Cancel** under it; Colab then carries on with the rest.
   If a box asks to give the notebook access to a secret, click **Grant access**.
   If Run all stops at a red error, click the next cell below the red one and choose
   **Runtime → Run cell and below**; the critic will still see the rest.
4. **Open Gemini** there (spark icon, bottom centre) and start a new chat (click **⋮** at the top right of the Gemini panel, then **New chat**, then **Yes**).
   A fresh chat has not seen your reasoning and has no work of its own to defend.
   If a box *Review custom instructions* appears, click **Use custom instruction**.
5. `PROMPT` **Paste the prompt** and press Enter. If you use another AI because Gemini
   stopped, it cannot see your notebook. First paste the prompt into the other AI's chat box and
   do not press Enter yet. Then, in your Lab 8 notebook, click **+ Code**, type
   `show.pipeline_code()` and run it: a box appears with the code of your six
   functions (the five stages and `pipeline`), the versions in use now. Click inside the box, press Ctrl+A (Cmd+A on a
   Mac) and Ctrl+C. Back in the other AI's chat box, click at the end of the prompt,
   press Shift+Enter (a new line, not send), paste the code, then press Enter.
6. `VERIFY` **Read the answer.** You should see one answer per probe, 1 to 9, each
   with PASS, FAIL or UNSURE and a one-line reason. If Gemini adds a new code cell
   with a green edge, click **Cancel**: the critic reviews, it does not change your
   code. Ignore the blue suggestion chips.
7. **Come back to this notebook** (its browser tab) and fill in Task 2.

```
Rules: Be adversarial but specific: your job is to find real defects, not to be
encouraging. Flag only defects that affect correctness or the checks below; do
not suggest style changes or extra features. Do not write or change code.

Task: You are reviewing a student data pipeline for a module on automated
decision systems. Check for each of these and say PASS, FAIL or UNSURE with a one-line reason:

1. Silent row loss: does any step drop rows without accounting for them?
2. Sentinel values: are non-measurement codes (-999, 0, 9999) handled?
3. Zero vs missing: are zeros that mean "no reading" distinguished from real zeros?
4. Time index: is the data on a regular grid before any .shift()?
5. Horizon mismatch: are the model and baseline scored at the same horizon?
6. Lookahead bias: does any signal use information from the future?
7. Baseline honesty: is the baseline the strongest one a sensible person would use?
8. Response layer: is the alert debounced and rate-limited?
9. Failure mode: what happens if the data source returns nothing?

For each FAIL give the specific line and what you would change.

My pipeline is the functions ingest, clean, analyse, decide, respond and
pipeline defined in this notebook.
```

## Task 2 — Record every finding, then answer it · `VERIFY`

One row per probe. **Your verdict column is what gets marked.** Use the table on
the Lab 9 sheet, or this one:

1. **Double-click the table** in the next cell (it starts `| # | Probe |`). Type
   between the `|` marks and keep the marks.
2. **Critic's finding:** fill this column for all nine rows first, before you open
   any new Gemini chat (a new chat hides the critic's answer). Copy its PASS, FAIL or UNSURE and its one-line reason. If it
   skipped a probe, write *not mentioned*. Then compare with your plan in 1A: did the
   critic FAIL the probe you predicted?
3. **My verdict:** leave all three words for now; you choose in step 5, one row at a time.
4. `PROMPT` `VERIFY` **Get the evidence, one row at a time.** Go to the Lab 8 tab, start a new chat (click **⋮** at the top right of the Gemini panel, then **New chat**, then **Yes**), and type *Which line in this notebook answers this question:*
   followed by that row's question from the critic prompt in Task 1B (for example *does
   any step drop rows without accounting for them?*), then *Quote the line and the
   printed number that shows it. End with one word: problem yes or problem no.*
   Press Enter. If Gemini adds a green cell, click **Cancel**. If it quotes no line,
   type once in the same chat *Quote the exact line from this notebook and the number
   it printed.* Then do step 5 for this row straight away, while Gemini's answer is on screen (the next new chat
   hides it). Then repeat steps 4 and 5 for the next row, until all nine are done.
5. `VERIFY` **Choose the verdict** with the table *Verdicts A, B, C* below: take the
   first row that fits, keep its verdict word in your table, and delete the other two
   words in that row. Write one short line per cell, with no `|` in it and no line
   breaks.
6. **Press Shift+Enter** to see the table again.

**How to choose a verdict**, on three made-up findings (it plays slowly, about a minute):
![Three made-up findings. Probe 1, FAIL, but the row count is 17,280 before and after: they disagree, C, Reject. Probe 5, UNSURE, no line quoted: A, Unsure. Probe 8, FAIL, and the alert fires 12 times an hour: they agree, B, Accept](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab09_verdicts.gif)

**Verdicts A, B, C**, checked in this order:

| | What you see | Verdict | Evidence to write |
|---|---|---|---|
| **A** | Gemini quoted no line or gave no yes or no; or the critic said UNSURE or *not mentioned* | **Unsure** | that probe's *would need* line from the list below |
| **B** | They agree: critic PASS and Gemini *problem no*, or critic FAIL and Gemini *problem yes* | **Accept** | for a PASS: *critic said PASS; I agree*. For a FAIL: leave it empty now; after Task 3, write what you changed |
| **C** | They disagree: critic PASS and Gemini *problem yes*, or critic FAIL and Gemini *problem no* | **Reject** | the line or number Gemini quoted |

If Gemini quoted a line but no printed number, its yes or no still decides: B or C.

The *would need* lines for A, one per probe:
- 1: *would need: the row count before and after clean*
- 2: *would need: how many -999 values are left after clean*
- 3: *would need: how many zeros there are before and after clean*
- 4: *would need: the most common gap between rows after clean*
- 5: *would need: the horizon each forecast was scored at*
- 6: *would need: the decisions with and without the last month of data*
- 7: *would need: the baseline's score next to the model's*
- 8: *would need: the alert count with and without the debounce*
- 9: *would need: what pipeline() prints when ingest returns no rows*

| # | Probe | Critic's finding | My verdict | Evidence |
|---|---|---|---|---|
| 1 | Silent row loss | | Accept / Reject / Unsure | |
| 2 | Sentinel values | | Accept / Reject / Unsure | |
| 3 | Zero vs missing | | Accept / Reject / Unsure | |
| 4 | Time index | | Accept / Reject / Unsure | |
| 5 | Horizon mismatch | | Accept / Reject / Unsure | |
| 6 | Lookahead bias | | Accept / Reject / Unsure | |
| 7 | Baseline honesty | | Accept / Reject / Unsure | |
| 8 | Response layer | | Accept / Reject / Unsure | |
| 9 | Failure mode | | Accept / Reject / Unsure | |

Task 3 is only for FAILs you accepted.
**Reject** = it is wrong, and here is the evidence that it is wrong.
**Unsure** = I could not tell; here is what I would need to check. A PASS from
the critic still gets a row, filled in as step 5 says.

For example, on Track 2, a row that says *Reject: the row count drops by 74, but
Lab 8 Task 3A explains it: 140 repeated times removed, 66 empty slots of 11 June
added as blank rows* is worth more than one that says *Accept*.

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

Your rows depend on your own Lab 8 pipeline, so these are what a good table
contains. Every one of the nine rows has the critic's word and reason, exactly one
verdict word left, and something in Evidence (a FAIL you accepted gets its evidence after Task 3). If the critic said UNSURE or *not
mentioned*, the verdict stays *Unsure* and the evidence is that row's *would need*
line, for example row 5: *would need: the horizon each forecast was scored at*. A
*Reject* names a line or a number, as in the Track 2 example under the table: *the
row count drops by 74, but Lab 8 Task 3A explains it*.
The common wrong table accepts every FAIL without checking it, and after Task 3
still has nothing in Evidence but the critic's own words. A PASS you agree with may say *critic said PASS; I
agree*, as step 5 allows. A FAIL you agree with gets what you changed, after Task 3.
A FAIL you reject needs a line or a number that shows the critic is wrong, because
critics are confidently wrong too.
</details>

## Task 3 — Fix the FAILs you accepted, then check again · `REFINE` `GENERATE` `VERIFY`

*Topic 9: three warning signs in a test the AI writes: it tests the wrong thing, a
check was changed so that it passes, or it says "passed" with no numbers.*

You have about 30 minutes: fix every **FAIL** you accepted, the most serious first.
Do steps 1 to 4 once for each **FAIL** finding you **accepted** (not the PASSes you agreed with),
then steps 5 and 6 once at the end:

If you accepted no FAIL, skip steps 1 to 4: write *No FAIL accepted* in the box
below, then do steps 5 and 6.

1. `REFINE` **Say the hidden rule first.** The fix prompt below ends with one
   sentence hidden: what the fix must leave exactly as it is, so that your evidence
   can still be trusted. **Before you open it, say your answer.**

<details><summary><b>Show the prompt</b> (click, then copy all of the prompt; do not press Enter yet)</summary>

> **Rules:** Explain in plain words what the fix does.
>
> **Task:** Fix the finding written under this message. Then write a test that shows
> the fix works: print the row count before and after, and the smallest and largest
> value of the columns it touches. The fix must not change or delete any existing
> check.

**The last sentence was the hidden one.** It says two things. Did your answer say both?

- **change**: a check can be loosened until it passes. Topic 9's example: a check
  for at least 17,354 rows, loosened to "at least 17,000", passes while 95 rows are
  still lost;
- **delete**: a deleted check prints nothing, so its FAIL simply disappears.

Your checks are your evidence. A fix only counts if the same checks still run.
</details>

2. `REFINE` **Paste it in the Lab 8 tab, and add the finding under it.**
   1. Switch to the Lab 8 tab. In Gemini, start a new chat (click **⋮** at the top
      right of the Gemini panel, then **New chat**, then **Yes**).
   2. Paste the prompt into the chat box (Ctrl+V, or Cmd+V on a Mac). Do not press
      Enter yet.
   3. Click at the end of the prompt and press Shift+Enter (a new line, not send).
   4. Type the finding in a few words, from the *Critic's finding* column of your
      Task 2 table, for example *clean() drops the outage rows without counting them*.
   5. Press Enter. Gemini adds a new code cell with a green edge to your Lab 8
      notebook. Do not click **Accept and run** yet: do step 3 first.

3. `GENERATE` `VERIFY` **Read the green cell's code before accepting.**
   1. Look for any line that starts with `check.`, and for any advice to delete one.
   2. If there is none (the usual case), click **Accept and run**.
   3. If there is one, click **Cancel** and type in the same chat *Do the same fix
      without changing or deleting any line that starts with check.* Then read the
      new green cell the same way.
   4. If, after it runs, Gemini shows red and green lines inside another of your Lab 8
      cells, read those the same way. If no `check.` line was changed or removed,
      click **Accept and run**; otherwise click **Cancel**.
   5. Note yes or no: did the fix touch a `check.` line? It is question 2 in step 4.
4. `VERIFY` **Judge its test, not only its answer.** Back in this notebook, in the box
   below, answer three questions for each fix:
   1. What did the test actually check? (the test lines in Gemini's green cell;
      *"it returns a table"* proves nothing)
   2. Did it touch a `check.` line? (your note from step 3)
   3. Did it print numbers, or only say "passed"? (the output under the green cell;
      a pass with no numbers is a claim, not evidence)
5. `VERIFY` **Run your Lab 8 Task 3 cells again**: first the cell that prints rows removed and
   added, then the check cell under it (the one with `EXPLANATION =`). If the check
   prints FAIL for `rows_accounted_for`, the numbers no longer match your
   explanation: double-click inside that check cell, replace the text between the quotes after
   `EXPLANATION =` with the new numbers, written the same way as in Lab 8 Task 3
   step 1 (number, removed or added, reason; joined with semicolons), and run the
   check cell again. If any other check fails, or you see red error text, the fix broke
   something: write that in the box below, put your hand up, and show the tutor. Copy
   the PASS and FAIL lines into the box.
6. **Go back to the Task 2 table** and write what you changed in the Evidence
   column of each FAIL you accepted.

If you accepted no FAIL, write *no fixes* and the reason in the box below, and go on
to the next section.

**Each fix, and what its test really showed:**

**My Lab 8 checks after the fixes (copy the PASS / FAIL lines):**

*(Your answer box: double-click this cell. Type your answer here.)*

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

Each fix needs three things: what the test actually checked, whether it touched a
`check.` line, and whether it printed numbers. Example, for the notebook's sample
finding *clean() drops the outage rows without counting them*: the test printed the
row count before and after the fix, and the smallest and largest value of the
column. No `check.` line changed. It showed numbers, so it counts as evidence. Your
counts depend on your own data. The common weak answer is *the test passed* or *it
returns a table*: a pass with no numbers is only a claim. For the second box, copy
the real PASS and FAIL lines. A FAIL on `rows_accounted_for` usually means you have
not updated `EXPLANATION =` with the new counts yet. It does not by itself mean the
fix broke something. If you accepted no FAIL, the answer is *no fixes* and the
reason.
</details>

## The question the critic cannot answer

An AI critic can check your code. It cannot tell you **what happens on the days
your system is wrong, and who pays for it.** That question is the Week 14 record
box on your CA3 Build 3 sheet, and it goes straight into your CA3 reflection. You
write it next week; for now, note one idea in the *Decision* box at the end.

---
## Read before you run

A Track 5 group asked the critic to check their weekly activity alert. The critic said PASS on “zero versus missing”. The code is below; its last lines print the count both ways.

The code is switched off: every line starts with `#`, so it cannot run yet.

1. **Read it. Do not run it.** Answer questions 1 and 2 in the box under the code
   cell, or on the sheet.
2. **Ask Gemini to explain it.** Click in the code cell, press Ctrl+A then Ctrl+C
   (Cmd on a Mac). In Gemini, start a new chat (**⋮** › **New chat** › **Yes**),
   paste the code into the chat box, add *Explain this code in plain words.* and
   press Enter. If Gemini offers a green cell or red and green lines, click
   **Cancel**: this step only explains. Check whether the explanation points out the problem.
   Often it does not.
3. **Then run it.** Delete the `# ` at the start of each line below the first line,
   run the cell, and see whether you were right.

**Question 1.** Was the critic right? Eleven days in this file have `steps` equal to 0 because the watch was not worn. What do those days do to `weekly`, and to `their_df.steps.mean()`?

**Question 2.** Predict which of the two printed counts is larger, and why. At step 3, run it and compare.

<details><summary>Hint 1: where to look</summary>

Inside `rolling().mean()`, is a 0 treated like a NaN, or like a reading?

</details>
<details><summary>Hint 2: closer to the answer</summary>

Like a reading. A drawer day pulls the weekly average and the overall mean down. The two printed counts show how much the zeros matter.

![The critic said PASS. With the zeros left in, the 7-day average drops under the alert line after every drawer day: 28 days flagged. With the zeros as NaN: 17. 13 alerts came only from days the watch was off](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab09_pass_wrong.gif)

</details>

An explanation that reads well is not evidence the code is right. 

**Optional, to fix it:** in the same chat, type *Fix the problem in this code that
Hint 2 describes. Say in one sentence what you changed.* If that one change matches
Hint 2, click **Accept and run** (accepting is right here: you asked for the fix). If
it changed other things too, click **Cancel** and type *fix only that, nothing else*.
You should see: the first count drops from about 28 to about 17, the same as the second.

In [ ]:
# ---- Every line below this one is the code. Do steps 1 and 2 first. For step 3, delete the "# " at the start of each line below, then run the cell. ----
# their_df = pd.read_csv("smart_self_daily.csv", parse_dates=["date"])
# weekly = their_df.steps.rolling(7).mean()
# low_activity = weekly < 0.7 * their_df.steps.mean()
# print("days flagged, zeros left in       :", int(low_activity.sum()))
# st = their_df.steps.replace(0, float("nan"))
# print("days flagged, zeros treated as NaN:", int((st.rolling(7, min_periods=4).mean() < 0.7 * st.mean()).sum()))

**Answer to question 1:**

**Answer to question 2:**

**Did Gemini's explanation mention the fault? (yes / no, and the sentence):**

**Was I right, once I ran it?**

---
## Quiz yourself before you leave (5 minutes)

The checks tested the code. This quiz tests what you understood. In Gemini, start a new chat
(**⋮** › **New chat** › **Yes**), paste this into the chat box and press Enter:

> **Rules:** Ask one short question at a time, in plain words. Take the examples from the code and outputs in this notebook. Wait for my answer. Then say in one sentence whether I am right, and why. Do not give the next question until I answer.
>
> **Task:** Quiz me on this lab: five questions on these ideas: what an adversarial review is for; why a critic's PASS is not evidence; how to defend correct work against a wrong finding; why you run the critic in a new chat, not the one that wrote the code; the question the critic cannot answer: who pays when the system is wrong. After the fifth, give my score out of 5 and the one idea I should revise, in one line.

Gemini asks one question at a time in the chat panel; answer in the chat box, in
your own words, without looking back. Ignore the suggested replies Gemini shows. If you
use ChatGPT, Claude or Copilot instead, first delete the sentence *Take the examples from
the code and outputs in this notebook.*: they cannot see your notebook.

Then write your score and the idea to revise below. Your tutor asks for them.

**Quiz score:** ___ / 5

**The idea I should revise, in my own words:**

*(Your answer box: double-click this cell. Type your answer here.)*

---
## Before you leave

Double-click the box cell below and type there, or write on the last page of the Lab 9
sheet. The findings table must have a verdict and evidence on **every** row.

- **Mistake caught:** in the box below, write the critic's mistake from a finding
  you rejected, in the box's three parts: what it claimed · what
  was really there · how you showed it. If you rejected none, use a finding you were
  unsure about instead: what it claimed · what you could not tell · what you would check.
- **Decision:** which findings you fixed and why, and one idea about who pays when
  your system is wrong.

**Reality check:** if you accepted every finding without pushing back on any, you
probably have not read them closely enough. Critics can be confidently wrong, so a
table with no Reject and no Unsure needs a second look.

**Mistake caught:** *(three short lines: 1. what the critic claimed, 2. what was really there, 3. how I showed it. If you rejected no finding: 1. a finding I was unsure about, 2. what I could not tell, 3. what I would check.)*

**Decision:** *(what I decided · why · the evidence)*

*(Your answer box: double-click this cell. Type your answer here.)*